In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
#ucčitavanje podataka
df = pd.read_csv(
    r"C:\Users\lovro\Desktop\hackatoni\LUMEN_DS.csv",
    sep="|",
    quotechar='"',
    encoding="utf-16",
)

In [ ]:
df = df[~df["Item Code"].isnull()]
codes = df["Item Code"].astype("string")


unique_codes = pd.unique(codes.dropna())
item2idx = {code: int(idx) for idx, code in enumerate(unique_codes)}

# Map to integer ids
df["item_idx"] = codes.map(item2idx)

In [ ]:
items = np.unique(df["item_idx"])

for i, item in enumerate(items):

    item_df = df[df["item_idx"] == item]

    print(f"price mean for item {item}: {item_df['Invoiced price'].mean()}, std: {item_df['Invoiced price'].std()}")
    print(f"quantity mean for item {item}: {item_df['Ordered qty'].mean()}, std: {item_df['Ordered qty'].std()}")
    print(f"cost of part for item {item}: {item_df['Cost of part'].mean() }, std: {item_df['Cost of part'].std()}   ")
    print(f"number of purchases for item {item}: {len(item_df)}")

In [ ]:
for i in df.columns:
    if df[i].dtype == "object":
        df[i][df[i].isna()] = "UNKNOWN"

In [ ]:
df.info()

In [ ]:
"""
Analyzing one invoice #
zakljucak je da je to jedan racuna s vise linija
isti item moze biti u vise linija
dolje je proucavan odnos cijene i qty i item codea i to nismo nasli povezanost
vecinom cijena za iste iteme bude ista, ponekad bude duplo visa na nekim.
"""

In [ ]:
inv, count = np.unique(df["Invoice #"], return_counts=True)
sorted_indices = np.argsort(count)[::-1]
inv = inv[sorted_indices]
count = count[sorted_indices]
print(inv, count)  
print(count[count > 1].sum())
print(len(count))


In [ ]:
inv, count = np.unique(df["Item Code"], return_counts=True)
print(len(inv))

In [ ]:
count[count < 5].sum()

In [ ]:
a = df[df["Invoice #"] == inv[1]]
d, c = np.unique(a["Invoice Line #"], return_counts=True)
d1, c1 = np.unique(a["Order Date"], return_counts=True)
d2, c2 = np.unique(a["Item Code"], return_counts=True)  
d3, c3 = np.unique(a["Invoiced price"], return_counts=True) 
d4, c4 = np.unique(a["Invoiced qty (shipped)"], return_counts=True)                  

c = np.sort(c)[::-1]
c1 = np.sort(c1)[::-1]
c2 = np.sort(c2)[::-1]
c3 = np.sort(c3)[::-1]
c4 = np.sort(c4)[::-1]
print(c)
print(c1)
print(c2)
print(c3)
print(c4)

In [ ]:
b = a.loc[:,["Item Code", "Invoiced price", "Invoiced qty (shipped)"]]
b.sort_values(by=["Item Code"], ascending=False, inplace=True)
np.array(b)

In [ ]:
""""
analyzing per costumer

"""

In [ ]:
people, count = np.unique(df["CustomerID"], return_counts=True)
sorted_indices = np.argsort(count)[::-1]
people = people[sorted_indices]
count = count[sorted_indices]
print(people, count)  
print(count[count > 1].sum())
print(len(count))


In [ ]:
num_invoices_per_customer = []
num_lines_per_customer = []
for customer in people:
    num_invoices = len(df[df["CustomerID"] == customer]["Invoice #"].unique())
    num_invoices_per_customer.append(num_invoices)
    num_lines_per_customer.append(len(df[df["CustomerID"] == customer]))    

plt.boxplot(num_invoices_per_customer)

In [ ]:
a = np.array(num_invoices_per_customer)
b = np.array(num_lines_per_customer)
print((b < 10).sum())

In [ ]:
#####
####
#####
#








#####

In [ ]:
" dalje analziramo za jedan item kako se ponsaju varijable za njega"

In [ ]:
item, count = np.unique(df["Item Code"], return_counts=True)
sorted_indices = np.argsort(count)[::-1]
item = item[sorted_indices]
count = count[sorted_indices]
print(item, count)  
print(count[count > 1].sum())
print(len(count))


In [ ]:


a = df[df["Item Code"] == item[0]]
d1, c1 = np.unique(a["CustomerID"], return_counts=True)
d3, c3 = np.unique(a["Invoiced price"], return_counts=True) 

sorted_indices = np.argsort(c3)[::-1]
c3 = c3[sorted_indices]
d3 = d3[sorted_indices]
sorted_indices = np.argsort(c1)[::-1]
c1 = c1[sorted_indices]
d1 = d1[sorted_indices] 

print(d3)
print(c3)
plt.scatter(c3,d3)
print(c1)
print(d1)

In [ ]:
costumer, count = np.unique(df["CustomerID"], return_counts=True)
sorted_indices = np.argsort(count)[::-1]
costumer = costumer[sorted_indices]
count = count[sorted_indices]
print(costumer, count)  
print(count[count > 1].sum())
print(len(count))


In [ ]:
a = df[df["CustomerID"] == costumer[6]]

item, count = np.unique(df["Item Code"], return_counts=True) ############ nije mu tu dolje jasno zasto je b prazno
sorted_indices = np.argsort(count)[::-1]
item = item[sorted_indices]
count = count[sorted_indices]

print((a["Item Code"] == item[0]).sum())
b = a[a["Item Code"] == item[0]]

d3, c3 = np.unique( b["Invoiced price"], return_counts=True) 

sorted_indices = np.argsort(c3)[::-1]
c3 = c3[sorted_indices]
d3 = d3[sorted_indices]

print(d3)
print(c3)
plt.scatter(c3,d3)

In [ ]:
"""
gledamo kako se product family i product groupovi ponasaju
isti group moze biti u vise familyja

"""

In [ ]:
def analyze_column_relationships(col1, col2):
    # printa u koliko se razlicitih col2 pojavljuje col1
    # vraca listu brojeva koliko se puta se pojavljuje col1 u col2
    temp = np.unique(df[col1])
    lib = {}
    for gr in temp:
        lib[gr] = 0 

    product_families = np.unique(df[col2])

    for product_fam in product_families:
        df_family = df[df[col2] == product_fam]
        product_groups = np.unique(df_family[col1])
        for product_group in product_groups:
            lib[product_group] += 1

    #for key in sorted(lib, key=lib.get):
    #    print(key, lib[key])

    # Descending order (largest to smallest values)
    vals = np.array(list(lib.values()))
    keys = np.array(list(lib.keys()))
    #for key in sorted(lib, key=lib.get, reverse=True):
    #    print(key, lib[key])


    sorted_indices = np.argsort(vals)[::-1]
    sorted_keys = keys[sorted_indices]
    sorted_vals = vals[sorted_indices]
    return  sorted_vals, sorted_keys
            


In [ ]:
analyze_column_relationships("Product family", "Product group")

In [ ]:
analyze_column_relationships("Product group", "Product family")

In [ ]:
counts, keys = analyze_column_relationships("Item Code", "Product group")
print(counts[:40], keys[:40])

In [ ]:
counts, keys = analyze_column_relationships("Item Code", "Product family")
print(counts[:40], keys[:40])

In [ ]:
# Nested loop: Product Family -> Product Group -> ItemCode (visualize average prices)
import matplotlib.pyplot as plt

product_families = np.unique(df["Product family"])

for product_fam in product_families:
    df_family = df[df["Product family"] == product_fam]
    product_groups = np.unique(df_family["Product group"])
    
    for prod_group in product_groups:
        df_group = df_family[df_family["Product group"] == prod_group]
        
        # Calculate average prices by ItemCode
        avg_prices = df_group.groupby("Item Code")["Invoiced price"].mean().sort_values(ascending=False)
        
        items = np.unique(df_group["Item Code"])
        if(items.shape[0] >= 1):
            print(avg_prices.shape)
            plt.plot(avg_prices)
            plt.show()


In [ ]:
a = df[df["Invoice #"] == inv[1]]
d, c = np.unique(a["Invoice Line #"], return_counts=True)
d1, c1 = np.unique(a["Order Date"], return_counts=True)
d2, c2 = np.unique(a["Item Code"], return_counts=True)                   
c = np.sort(c)[::-1]
c1 = np.sort(c1)[::-1]
c2 = np.sort(c2)[::-1]
print(c)
print(c1)
print(c2)

In [ ]:
a,c = np.unique(df["Invoice Line "], return_counts=True)
sorted_indices = np.argsort(c)[::-1]
a = a[sorted_indices]  
c = c[sorted_indices]
print(a,c)

In [ ]:
for data in df["Invoice Line #"]:
    if not isinstance(data, int):
        print(data)

In [ ]:
product_family = np.unique("Product family")
for product_fam in product_family:
    df_temp = df[df["Product family"] == product_fam]
    product_group = np.unique(df_temp["Product group"])
    for prod_g in product_group:
         

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    sparse_output=False,  # Set to True for memory efficiency with many categories
    handle_unknown='ignore',  # Handle unseen categories in test data
)

encoded_array = encoder.fit_transform(df[['Manufacturing Region']])

# Get feature names
feature_names = np.array([f'Manufacturing Region_{i}' for i,cat in enumerate(encoder.categories_[0])])

# Convert to DataFrame
df_encoded = pd.DataFrame(
    encoded_array,
    columns=feature_names,
    index=df.index
)

# Combine with original numerical columns
numerical_cols = df.select_dtypes(include=['number']).columns
df_final = pd.concat([df[numerical_cols], df_encoded], axis=1)


In [ ]:
people = df_final["CustomerID"].unique()
person = people[0]
person_data = df_final[df_final["CustomerID"] == person].copy()

df_final = df.copy()

for i in range(len(feature_names)):


In [ ]:
for column in categorical_cols:
    encoder = OneHotEncoder(
        sparse_output=False,  # Set to True for memory efficiency with many categories
        handle_unknown='ignore',  # Handle unseen categories in test data
    )

    encoded_array = encoder.fit_transform(df[[column]])

    # Get feature names
    feature_names = np.array([f'{column}_{i}' for i,cat in enumerate(encoder.categories_[0])])

    # Convert to DataFrame
    df_encoded = pd.DataFrame(
        encoded_array,
        columns=feature_names,
        index=df.index
    )
    df_final = pd.concat([df_final, df_encoded], axis=1)




In [ ]:
df_final.columns

In [ ]:
for i in df.columns:
    if df[i].dtype == "object":
        df[i][df[i].isna()] = "UNKNOWN"

In [ ]:
a,c  = np.unique(df["Customer industry"], return_counts=True)
a1,c1  = np.unique(df["Customer Region"], return_counts=True)

In [ ]:
print(a)
print(c)

In [ ]:
people = df["CustomerID"].unique()
person_data = df[df["CustomerID"] == person].copy() 
person_data["Top Customer Group"].values[0]

In [ ]:
other, star = 0, 0
people = df["CustomerID"].unique()
for i, person in enumerate(people): # iteriramo po svim ljudima
    person_data = df[df["CustomerID"] == person].copy() 
    
    if person_data.shape[0] >0 and person_data["Top Customer Group"].values[0] == "STAR":
        star += 1
    else:
        other += 1
print(f"STAR: {star}, OTHER: {other}")

In [ ]:
# from numpy import NaN

# df['Order Date_temp'] = pd.to_datetime(df['Order Date'], errors='coerce')

# valid_date_rows = df[~df['Order Date_temp'].isna()]

# min_total = valid_date_rows["Order Date_temp"].min()


# people = df["CustomerID"].unique()

# data_points = np.zeros((people.shape[0], 10))


# for i, person in enumerate(people):
    
#     n = 0
#     while n == 0 :
#         person_data = df[df["CustomerID"] == person]
#         n = person_data.shape[0]

#     person_data['Order Date_temp'] = pd.to_datetime(person_data['Order Date'], errors='coerce')

#     invalid_date_rows = person_data[person_data['Order Date_temp'].isna()]
#     m = len(invalid_date_rows)
#     n = person_data.shape[0]

#     if m == 0:
#         min,max = person_data["Order Date"].min(), person_data["Order Date"].max()
#         date1 = pd.to_datetime(min)
#         date2 = pd.to_datetime(max)
#         days_diff = (date2 - date1).days
#         last_order = (date1 - min_total).days
#     else:
#         days_diff = NaN
#         last_order = NaN


#     avg_price = person_data["Invoiced price"].mean()
#     avg_count_ordered = person_data["Ordered qty"].mean()
#     avg_count_delivered = person_data["Invoiced qty (shipped)"].mean()

#     corr_price_ordered = person_data["Invoiced price"].corr(person_data["Ordered qty"])
#     corr_price_delivered = person_data["Invoiced price"].corr(person_data["Invoiced qty (shipped)"])
#     corr_qty = person_data["Ordered qty"].corr(person_data["Invoiced qty (shipped)"])

#     gm = person_data["GM%"].mean()
#     data_points[i] = [days_diff, last_order, n, avg_price, avg_count_ordered, avg_count_delivered,
#                       corr_price_ordered, corr_price_delivered, corr_qty, gm]               

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler



def remove_outliers_iqr(df, columns=None, multiplier=1.5):
    """
    uklanja retke koji imaju outlier prema IQR metodi
    gledaju se outlieri po stupcima columns
    """
    if columns is None:
        columns = df.columns
    
    df_clean = df.copy()
    outliers_mask = pd.Series(False, index=df.index)
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        col_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers_mask = outliers_mask | col_outliers
        
        print(f"{col}: {col_outliers.sum()} outliers ({col_outliers.sum()/len(df)*100:.1f}%)")
    
    print(f"\nTotal rows before removal: {len(df)}")
    print(f"Total outliers: {outliers_mask.sum()} ({outliers_mask.sum()/len(df)*100:.1f}%)")
    
    df_clean = df[~outliers_mask]
    print(f"Total rows after removal: {len(df_clean)}")
    
    return df_clean, outliers_mask


In [ ]:
#ovdje gledamo koliko svaki covjek/kompanija je napravio narudžbi i po toj mjeri micemo outliere, njih je 1%, njih posebno hendlat

n_orders_per_customer = df.groupby('CustomerID').size().reset_index(name='n_orders')
high_volume_threshold = n_orders_per_customer['n_orders'].quantile(0.99)  # Top 1%
high_volume_customers = n_orders_per_customer[
    n_orders_per_customer['n_orders'] > high_volume_threshold
]['CustomerID'].tolist()

print(f"High-volume customers (>{high_volume_threshold:.0f} orders): {len(high_volume_customers)} ({len(high_volume_customers)/len(n_orders_per_customer)*100:.1f}%)")

# Separate them
df_high_volume = df[df['CustomerID'].isin(high_volume_customers)]
df_regular = df[~df['CustomerID'].isin(high_volume_customers)]

In [ ]:
import numpy as np
import pandas as pd
from numpy import NaN
from sklearn.impute import SimpleImputer

def imputing(df):
    """
    vraca data points dta frame i np  array
    svaki redak je 10 mjera koje se izračunaju za neku osobu 
    mogli bismo i neke druge mjere, ove sam ja tako, po svojoj procjeni stavio, koje su mi se činile bitne

    """
    df['Order Date_temp'] = pd.to_datetime(df['Order Date'], errors='coerce')

    valid_date_rows = df[~df['Order Date_temp'].isna()]

    min_total = valid_date_rows["Order Date_temp"].min() # tu dobijemo najstariju kupovinu ukupno


    people = df["CustomerID"].unique()
    print(f"Total unique customers: {len(people)}")
    data_points = np.zeros((people.shape[0], 10))

    for i, person in enumerate(people): # iteriramo po svim ljudima
        
        
        person_data = df[df["CustomerID"] == person].copy()  
        
        n = person_data.shape[0]
        
        person_data['Order Date_temp'] = pd.to_datetime(person_data['Order Date'], errors='coerce')
        
        # gledamo datuma koji nisu ispravno upisani ili nedostaju
        invalid_date_rows = person_data[person_data['Order Date_temp'].isna()]
        m = len(invalid_date_rows)
        

        # u ovom bloku racunamo učestalost(frequency) kupovine, i posljednji put kada je osoba kupovala
        if m == 0 and n > 0:  
            try:
                min_date = person_data["Order Date_temp"].min()
                max_date = person_data["Order Date_temp"].max()
                
                if pd.notna(min_date) and pd.notna(max_date):
                    days_diff = (max_date - min_date).days
                    last_order = (min_date - min_total).days
                else:
                    days_diff = np.nan
                    last_order = np.nan
            except:
                days_diff = np.nan
                last_order = np.nan
        else:
            days_diff = np.nan
            last_order = np.nan
        
        
        #računamo prosječnu cijenu, brojnaručenih predmeta i broj dostavljenih predmeta
        avg_price = person_data["Invoiced price"].mean(skipna=True)
        avg_count_ordered = person_data["Ordered qty"].mean(skipna=True)
        avg_count_delivered = person_data["Invoiced qty (shipped)"].mean(skipna=True)
        
        #blok računa korelaciju broja naručenih podataka i cijene
        valid_pairs_price_ordered = person_data[["Invoiced price", "Ordered qty"]].dropna().shape[0]
        if valid_pairs_price_ordered >= 3:  # zahtjevamo barem tri podatka za korelaciju
            corr_price_ordered = person_data["Invoiced price"].corr(person_data["Ordered qty"])
        else:
            corr_price_ordered = np.nan
        
        
        #blok računa korelaciju broja dostavljenih podataka i cijene
        valid_pairs_price_delivered = person_data[["Invoiced price", "Invoiced qty (shipped)"]].dropna().shape[0]
        if valid_pairs_price_delivered >= 3:  # zahtjevamo barem tri podatka za korelaciju
            corr_price_delivered = person_data["Invoiced price"].corr(person_data["Invoiced qty (shipped)"])
        else:
            corr_price_delivered = np.nan
        
        
        #blok računa korelaciju broja naručenih podataka i broja dostavljenih
        valid_pairs_qty = person_data[["Ordered qty", "Invoiced qty (shipped)"]].dropna().shape[0]
        if valid_pairs_qty >= 3:  # zahtjevamo barem tri podatka za korelaciju
            corr_qty = person_data["Ordered qty"].corr(person_data["Invoiced qty (shipped)"])
        else:
            corr_qty = np.nan
        
        
        #sprema gm% to je u biti zarada: (prodajna cijena-cijena)/prodajna cijena 
        gm_data = person_data["GM%"].dropna()
        if len(gm_data) >= 3:
            gm = gm_data.mean()
        else:
            gm = np.nan
        
        # popuniti sve s nan
        data_points[i] = [days_diff if not pd.isna(days_diff) else np.nan,
                        last_order if not pd.isna(last_order) else np.nan,
                        n,
                        avg_price if not pd.isna(avg_price) else np.nan,
                        avg_count_ordered if not pd.isna(avg_count_ordered) else np.nan,
                        avg_count_delivered if not pd.isna(avg_count_delivered) else np.nan,
                        corr_price_ordered,  # Already handled NaN
                        corr_price_delivered,  # Already handled NaN
                        corr_qty,  # Already handled NaN
                        gm if not pd.isna(gm) else np.nan]

    # After building data_points, impute remaining NaNs with median
    print(f"NaN counts before imputation:")
    print(pd.DataFrame(data_points).isna().sum())

    # median imputer
    imputer = SimpleImputer(strategy='median')
    data_points_imputed = imputer.fit_transform(data_points)

    print(f"\nNaN counts after imputation:")
    print(pd.DataFrame(data_points_imputed).isna().sum())

    # genreira dataframe od podatka
    column_names = ['days_diff', 'last_order', 'n_orders', 'avg_price', 
                    'avg_ordered_qty', 'avg_delivered_qty',
                    'corr_price_ordered', 'corr_price_delivered', 
                    'corr_qty', 'gm']

    data_points_imputed_df = pd.DataFrame(data_points_imputed, columns=column_names)

    return data_points_imputed_df, data_points_imputed
# Example of running k-means on the imputed data:
# from sklearn.cluster import KMeans
# kmeans = KMeans(n_clusters=3, random_state=42)
# clusters = kmeans.fit_predict(data_points_imputed)

In [ ]:
a = np.unique(df_regular['CustomerID'])
a.shape

In [ ]:
data_points_imputed_df, data_points_imputed = imputing(df_regular)       

In [ ]:

# Remove outliers
columns_for_outlier_removal = ['avg_price', 'avg_ordered_qty', 'avg_delivered_qty', 
                                'days_diff', 'gm']  # Choose relevant columns


# ovaj dio makne pola podataka
#df_clean, outlier_mask = remove_outliers_iqr(data_points_imputed_df, columns_for_outlier_removal)


In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_points_imputed_df)



In [ ]:
k = 2 
min_size = 100

N = data_scaled.shape[0]


# Track original indices
active_idx = np.arange(N)
removed_idx = np.array([], dtype=int)

while True:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_scaled[active_idx])
    
    labels = kmeans.labels_
    unique, counts = np.unique(labels, return_counts=True)
    
    small_clusters = unique[counts < min_size]
    
    if len(small_clusters) == 0:
        break
    
    mask_small = np.isin(labels, small_clusters)
    
    # Move small cluster indices to removed
    removed_idx = np.concatenate([removed_idx, active_idx[mask_small]])
    
    # Keep only large clusters
    active_idx = active_idx[~mask_small]

# Final clustering on surviving points
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
kmeans.fit(data_scaled[active_idx])

final_labels_active = kmeans.labels_

# Assign removed points to nearest centroid
if len(removed_idx) == 0:
    final_labels = np.empty(N, dtype=int)
    final_labels[active_idx] = final_labels_active
else:
    removed_points = data_scaled[removed_idx]
    distances = kmeans.transform(removed_points)
    final_labels_removed = np.argmin(distances, axis=1)

    # Construct final label array
    final_labels = np.empty(N, dtype=int)
    final_labels[active_idx] = final_labels_active
    final_labels[removed_idx] = final_labels_removed


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score as sklearn_silhouette_score
from sklearn.preprocessing import StandardScaler

# Scale
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_points_imputed_df)
min_size = 100

def adjusted_kmeans(data_scaled, k, min_size=100):
    N = data_scaled.shape[0]


    # Track original indices
    active_idx = np.arange(N)
    removed_idx = np.array([], dtype=int)

    while True:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(data_scaled[active_idx])
        
        labels = kmeans.labels_
        unique, counts = np.unique(labels, return_counts=True)
        
        small_clusters = unique[counts < min_size]
        
        if len(small_clusters) == 0:
            break
        
        mask_small = np.isin(labels, small_clusters)
        
        # Move small cluster indices to removed
        removed_idx = np.concatenate([removed_idx, active_idx[mask_small]])
        
        # Keep only large clusters
        active_idx = active_idx[~mask_small]

    # Final clustering on surviving points
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_scaled[active_idx])

    final_labels_active = kmeans.labels_

    # Assign removed points to nearest centroid
    if len(removed_idx) == 0:
        final_labels = np.empty(N, dtype=int)
        final_labels[active_idx] = final_labels_active
    else:
        removed_points = data_scaled[removed_idx]
        distances = kmeans.transform(removed_points)
        final_labels_removed = np.argmin(distances, axis=1)

        # Construct final label array
        final_labels = np.empty(N, dtype=int)
        final_labels[active_idx] = final_labels_active
        final_labels[removed_idx] = final_labels_removed

    return final_labels, active_idx, removed_idx

In [ ]:
import numpy as np

def metrics(data_points, labels):
    k = labels.max() + 1
    
    # Compute centroids
    centroids = np.vstack([
        data_points[labels == i].mean(axis=0)
        for i in range(k)
    ])
    
    # Silhouette
    silhouette = sklearn_silhouette_score(data_points, labels)
    
    # Mean distance to assigned centroid
    mean_centroid_distance = np.mean(
        np.linalg.norm(data_points - centroids[labels], axis=1)
    )
    
    return silhouette, mean_centroid_distance

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

"""
 tu se računa klasični k means i silouhette score(to nisam ziher kaj je to je chat dao) 
 za raličit broj clustera da se odredi optimalni k
"""

column_names = ['days_diff', 'last_order', 'n_orders', 'avg_price', 
                'avg_ordered_qty', 'avg_delivered_qty',
                'corr_price_ordered', 'corr_price_delivered', 
                'corr_qty', 'gm']


# 2. Scale the data (crucial for k-means!)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_points_imputed_df)

# 3. Find optimal number of clusters using elbow method
inertias = []
silhouette_scores = []
K_range = range(2, 30)

for k in K_range:
    """
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_scaled)
    silhouette_score, inertia = metrics(data_scaled, kmeans.labels_)
    """
    #ovo je adjusted k means koji uvijek daje clustere koji imaju barem 100 članova
    final_labels, active_idx, removed_idx = adjusted_kmeans(data_scaled,k, min_size=100)
    silhouette_score, inertia = metrics(data_scaled, final_labels)
    
    inertias.append(inertia)
    silhouette_scores.append(silhouette_score)

# Plot elbow curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow method plot
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method for Optimal k')
axes[0].grid(True)

# Silhouette score plot
axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score for Optimal k')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Print silhouette scores
for k, score in zip(K_range, silhouette_scores):
    print(f"k={k}: Silhouette Score = {score:.4f}")

# 4. Choose optimal k (let's say based on silhouette score)


In [ ]:
data_scaled.shape

In [ ]:
cluster_labels, active_idx, removed_idx = adjusted_kmeans(data_scaled, 7, min_size=100)
np.unique(cluster_labels)

In [ ]:
# odabrao sam 11 tu sad dalje ide neka analiza i vizualizacija
optimal_k = 8
print(f"\nOptimal number of clusters: {optimal_k}")

print(data_scaled.shape)

# ovo je klasicni kmeans
"""
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(data_scaled)
"""
#ovo je adjusted k means koji uvijek daje clustere koji imaju barem 100 članova
cluster_labels, active_idx, removed_idx = adjusted_kmeans(data_scaled, optimal_k, min_size=100)


# Add cluster labels to your dataframe
labeled_data_points_imputed_df = data_points_imputed_df.copy()
labeled_data_points_imputed_df['Cluster'] = cluster_labels

# 6. Analyze cluster characteristics
print("\n=== Cluster Analysis ===")
print("\nCluster sizes:")
print(labeled_data_points_imputed_df['Cluster'].value_counts().sort_index())

# Calculate mean values for each cluster
cluster_means = labeled_data_points_imputed_df.groupby('Cluster').mean()
print("\nCluster means (scaled back to original units):")
print(cluster_means)

# 7. Visualize cluster profiles
# Create a heatmap of cluster characteristics
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_means.T, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', cbar_kws={'label': 'Mean Value'})
plt.title('Cluster Profiles - Mean Values')
plt.ylabel('Features')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

# 8. Visualize clusters using PCA for 2D visualization
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(data_pca[:, 0], data_pca[:, 1], 
                     c=cluster_labels, cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Customer Segments Visualization (PCA)')
plt.grid(True, alpha=0.3)
plt.show()

# 9. Analyze feature importance for clustering
# Look at cluster centers
cluster_centers = scaler.inverse_transform(final_kmeans.cluster_centers_)
cluster_centers_df = pd.DataFrame(cluster_centers, columns=column_names)
print("\nCluster Centers (original units):")
print(cluster_centers_df)

# 10. Save results
labeled_data_points_imputed_df.to_csv('customer_clusters.csv', index=False)
print("\nResults saved to 'customer_clusters.csv'")

# 11. Optional: Characterize each cluster
print("\n=== Cluster Characterization ===")
for cluster in range(optimal_k):
    print(f"\nCluster {cluster} (n={sum(cluster_labels==cluster)} customers):")
    cluster_data = labeled_data_points_imputed_df[labeled_data_points_imputed_df['Cluster'] == cluster]
    
    # Print key characteristics
    print(f"  Average days between orders: {cluster_data['days_diff'].mean():.1f}")
    print(f"  Average order value: ${cluster_data['avg_price'].mean():.2f}")
    print(f"  Average order quantity: {cluster_data['avg_ordered_qty'].mean():.1f}")
    print(f"  Average GM%: {cluster_data['gm'].mean():.1f}%")
    print(f"  Price-Order correlation: {cluster_data['corr_price_ordered'].mean():.3f}")

In [ ]:
"""
postoje invoice price 0 narudzbe, onda gm% bude nan i nemogu korelaciju gledati
ponekad je inovice price 0, a product cost nije, mogao bih onda gm% postaviti na -max
order qty i deliver qty se razlikuju
invoice dateovi neki su invalidni


nisam zadovoljan sve skupa jer postoje klasteri s po jednom osobom, možda su to ovi izdvojeni iz skupine na slici
"""

## Dva klju?na problema dataseta

Sljede?a dva grafa sa?imaju probleme koji su najbitniji za recommendation i validaciju: 
1. support po customerima je vrlo neujedna?en
2. itemi imaju izra?enu long-tail raspodjelu popularnosti


In [ ]:
# Problem 1: support po customerima je vrlo neujedna?en
customer_support = df.groupby("CustomerID").size().sort_values(ascending=False).reset_index(drop=True)
customer_rank = np.arange(1, len(customer_support) + 1)

plt.figure(figsize=(10, 6))
plt.plot(customer_rank, customer_support.values, linewidth=2)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Rank customera (log scale)")
plt.ylabel("Broj purchase lineova po customeru (log scale)")
plt.title("Neujedna?en support po customerima")
plt.grid(True, which="both", alpha=0.3)
plt.show()

print(f"Median lineova po customeru: {customer_support.median():.0f}")
print(f"90. percentil lineova po customeru: {customer_support.quantile(0.9):.0f}")
print(f"99. percentil lineova po customeru: {customer_support.quantile(0.99):.0f}")


In [ ]:
# Usporedba kumulativne raspodjele purchase lineova po customerima i itemima
customer_support = df.groupby("CustomerID").size().sort_values(ascending=False).reset_index(drop=True)
item_support = df.groupby("Item Code").size().sort_values(ascending=False).reset_index(drop=True)

cum_customer_share = np.arange(1, len(customer_support) + 1) / len(customer_support)
cum_customer_purchase_share = customer_support.cumsum() / customer_support.sum()

cum_item_share = np.arange(1, len(item_support) + 1) / len(item_support)
cum_item_purchase_share = item_support.cumsum() / item_support.sum()

plt.figure(figsize=(10, 6))
plt.plot(cum_customer_share, cum_customer_purchase_share, linewidth=2, label="Customeri")
plt.plot(cum_item_share, cum_item_purchase_share, linewidth=2, label="Itemi")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Idealno ravnomjerno")
plt.xlabel("Udio entiteta")
plt.ylabel("Kumulativni udio svih purchase lineova")
plt.title("Neujednačenost udjela kupnji po customeriam i itemima")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

top_10_pct_customers = max(1, int(np.ceil(0.10 * len(customer_support))))
top_10_pct_items = max(1, int(np.ceil(0.10 * len(item_support))))
share_top_10_pct_customers = customer_support.iloc[:top_10_pct_customers].sum() / customer_support.sum()
share_top_10_pct_items = item_support.iloc[:top_10_pct_items].sum() / item_support.sum()

print(f"Top 10% customera nosi {share_top_10_pct_customers:.2%} svih purchase lineova.")
print(f"Top 10% itema nosi {share_top_10_pct_items:.2%} svih purchase lineova.")
